# CourtLens 하이라이트 정제 모델 (Colab)

룰 베이스로 1차 가공된 KBL 하이라이트 클립에서 불필요한 구간을 잘라냅니다.

- **기준 구축 (NBA 하이라이트만 필요, 풀경기 불필요)**: 공식 하이라이트 세그먼트의 CLIP 임베딩을 '기준 은행'으로 저장
- **추론 (KBL)**: 1차 클립의 각 1초가 은행과 얼마나 비슷한지(kNN 유사도) 점수화 → 임계값 컷 → 정제된 클립

런타임 유형을 **GPU(T4)** 로 설정하고 실행하세요.

In [ ]:
# 1. 코드 및 의존성 설치
!git clone -b ai-model https://github.com/gomja0124/likelion_CourtLens.git
%cd likelion_CourtLens/ai/highlight_refiner
!pip -q install -r requirements.txt

In [ ]:
# 2. CLIP 인코더 로드
import torch
from src.features import load_clip_encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
clip_model, clip_preprocess = load_clip_encoder(device)

In [ ]:
# 3. NBA 하이라이트 다운로드 (5~10개 권장, 풀경기는 필요 없음)
# 주의: 내부 연구/데모 용도로만 사용, 재배포 금지
from src.data_collect import download_videos

HIGHLIGHT_URLS = [
    # NBA 공식 하이라이트 URL을 채우세요
    # "https://www.youtube.com/watch?v=...",
]

highlight_files = download_videos(HIGHLIGHT_URLS, "data/highlights")
print(len(highlight_files), "개 다운로드됨")

In [ ]:
# 4. 하이라이트 기준 은행 구축 (특징 추출 — 영상 양에 따라 5~20분)
from src.knn_scorer import build_bank

scorer = build_bank(
    [str(p) for p in highlight_files],
    clip_model, clip_preprocess,
    out_path="checkpoints/highlight_bank.npz",
    device=device,
)

In [ ]:
# 5. KBL 1차 클립 업로드 → 정제 실행
from google.colab import files
from src.refine import refine_clip, plot_scores

uploaded = files.upload()  # 1차 가공된 KBL 클립(mp4) 업로드
kbl_clip = list(uploaded.keys())[0]

result = refine_clip(kbl_clip, scorer, clip_model, clip_preprocess, device=device)
print(f"{result['duration']}초 → {result['kept_duration']}초 (제거율 {result['removed_ratio']*100:.0f}%)")
print("keep 구간:", result["intervals"])
plot_scores(result)

In [ ]:
# 6. 결과 다운로드 (은행 파일도 함께 받아두면 로컬 refine에 재사용 가능)
files.download(result["output"])
files.download("checkpoints/highlight_bank.npz")

## 임계값 튜닝

잘리는 게 너무 많으면 `hi`/`lo`를 낮추고, 쓸데없는 구간이 남으면 올리세요. 은행 재구축 없이 재실행 가능합니다.

```python
result = refine_clip(kbl_clip, scorer, clip_model, clip_preprocess, device=device, hi=0.55, lo=0.4)
```

다음 세션에서는 `from src.knn_scorer import KnnScorer; scorer = KnnScorer.load("checkpoints/highlight_bank.npz")`로 바로 불러옵니다.

## (선택) 풀경기 영상을 확보한 경우

풀경기 영상이 있으면 음성 샘플을 추가한 이진 분류 학습으로 정확도를 더 올릴 수 있습니다 —
`src/dataset.py`(build_dataset) + `src/train.py`(train_scorer) 사용. README 참고.